In [1]:
import pandas as pd
import pathlib
import random

BASE_DIR = pathlib.Path().resolve().parent
DATABASE_DIR = BASE_DIR / "datasets"
EXPORT_DIR = DATABASE_DIR / "export"
SPAM_DATASET_PATH = EXPORT_DIR / "spam-dataset.csv"

## 1. Convert Data into Vectors

In [2]:
df = pd.read_csv(SPAM_DATASET_PATH)
df.head()

,label,text,source
0,ham,"Go until jurong point, crazy.. Available only ...",sms-spam
1,ham,Ok lar... Joking wif u oni...,sms-spam
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,sms-spam
3,ham,U dun say so early hor... U c already then say...,sms-spam
4,ham,"Nah I don't think he goes to usf, he lives aro...",sms-spam


In [3]:
labels = df['label'].tolist()
texts = df['text'].tolist()
labels[100],texts[100]

('ham', "Please don't text me anymore. I have nothing else to say.")

In [4]:
label_legend = {"ham": 0, "spam" : 1}
label_legend_inverted = {f"{v}" : k for k,v in label_legend.items()}
label_legend_inverted

{'0': 'ham', '1': 'spam'}

In [5]:
labels_as_int = [label_legend[x] for x in labels]
# label_legend_inverted[str(labels_as_int[120])]

In [6]:
random_idx = random.randint(0, len(labels))
labels_as_int[random_idx]

1

In [7]:
assert texts[random_idx] == df.iloc[random_idx].text
assert labels[random_idx] == df.iloc[random_idx].label
assert label_legend_inverted[str(labels_as_int[random_idx])] == df.iloc[random_idx].label

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [9]:
MAX_NUM_WORDS = 280

In [10]:
tokenizer = Tokenizer(num_words = MAX_NUM_WORDS) # keeps IDs 1 thro 279
tokenizer.fit_on_texts(texts)  # fit_on_texts learns the mapping
sequences = tokenizer.texts_to_sequences(texts) # texts_to_sequences applies it

In [11]:
tokenizer.sequences_to_texts(
    [[55, 66, 10, 123, 143, 204, 169, 77, 68, 187]]
)

['go only in n great world e there got wat']

In [12]:
word_index = tokenizer.word_index
word_index

{'i': 1,
 'to': 2,
 'you': 3,
 'a': 4,
 'the': 5,
 'and': 6,
 'my': 7,
 'u': 8,
 'is': 9,
 'in': 10,
 'this': 11,
 'me': 12,
 'it': 13,
 'for': 14,
 'of': 15,
 'on': 16,
 'out': 17,
 'your': 18,
 '\ufeff': 19,
 'have': 20,
 'so': 21,
 'that': 22,
 'check': 23,
 'are': 24,
 '2': 25,
 'call': 26,
 'if': 27,
 'but': 28,
 'can': 29,
 'just': 30,
 'now': 31,
 'not': 32,
 'be': 33,
 'at': 34,
 'will': 35,
 'do': 36,
 'or': 37,
 'like': 38,
 'get': 39,
 'with': 40,
 'up': 41,
 "i'm": 42,
 'we': 43,
 'no': 44,
 'love': 45,
 'ur': 46,
 'from': 47,
 'please': 48,
 'all': 49,
 'com': 50,
 'lt': 51,
 'gt': 52,
 'how': 53,
 'when': 54,
 'go': 55,
 '4': 56,
 'video': 57,
 'know': 58,
 'free': 59,
 'am': 60,
 'what': 61,
 'good': 62,
 'was': 63,
 'ok': 64,
 'time': 65,
 'only': 66,
 'then': 67,
 'got': 68,
 'its': 69,
 'song': 70,
 'come': 71,
 '39': 72,
 'youtube': 73,
 'new': 74,
 'br': 75,
 'as': 76,
 'there': 77,
 'day': 78,
 'want': 79,
 'he': 80,
 'one': 81,
 'www': 82,
 'by': 83,
 'amp': 84,
 

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
MAX_SEQ_LENGTH = 300

In [14]:
X = pad_sequences(sequences, maxlen= MAX_SEQ_LENGTH) # add padding so that all vectors have the same length
X

array([[  0,   0,   0, ...,  77,  68, 187],
       [  0,   0,   0, ...,   0,  64,   8],
       [  0,   0,   0, ...,   2, 110, 104],
       ...,
       [  0,   0,   0, ...,  15,   6, 137],
       [  0,   0,   0, ..., 180,  50,  50],
       [  0,   0,   0, ..., 190, 241,  19]],
      shape=(7528, 300), dtype=int32)

In [15]:
from tensorflow.keras.utils import to_categorical
import numpy as np

In [16]:
labels_as_int_arr = np.asarray(labels_as_int)
labels_as_int_arr

array([0, 0, 1, ..., 1, 1, 0], shape=(7528,))

In [17]:
# One-hot encode class labels: 0 → [1, 0], 1 → [0, 1]
# Use with 2 softmax outputs and categorical cross-entropy loss
y = to_categorical(labels_as_int_arr) 

## 2. Train Set Split

In [18]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.33, random_state= 42)

In [19]:
import pickle 

In [ ]:
training_data = {
    "X_train" : X_train,
    "X_test" : X_test,
    "y_train" : y_train,
    "y_test" : y_test,
    "max_words" : MAX_NUM_WORDS,
    "max_seq_length" : MAX_SEQ_LENGTH,
    "label_legend" : label_legend,
    "label_legend_inverted": label_legend_inverted,
}

tokenizer_json = tokenizer.to_json()